Good move — pinning down edge-case semantics _before_ coding is exactly the invariant-first discipline. Here's your next one.

---

## Problem: Count Paths in a Grid

You're standing at the **top-left** cell of a grid with $m$ rows and $n$ columns. You want to reach the **bottom-right** cell.

You may only move **right** or **down**, one cell at a time.

Write:

```python
def count_paths(m: int, n: int) -> int:
    # number of distinct paths from top-left to bottom-right
```

**Example:** a $2 \times 3$ grid (2 rows, 3 columns) has **3** paths.

---

## Edge-case semantics (stated, as requested)

- `count_paths(1, 1)` → **1**. A single cell: you're already at the destination. The empty path (no moves) counts as one valid path.
- `count_paths(1, n)` → **1** for any $n \geq 1$. One row means you can only go right; there's exactly one way.
- `count_paths(m, 1)` → **1** for any $m \geq 1$. One column, only go down, one way.
- `count_paths(0, n)` or `count_paths(m, 0)` → **0**. A grid with a zero dimension has no cells, so there's nowhere to stand and no path. _(You may or may not need this as an explicit base case — decide based on your recurrence.)_

---

## Constraints

- Pure recursion, no memoization, no loops.
- Same discipline as before: **write the recurrence and base cases as a comment first**, then the code.

---

## One nudge on the invariant

Don't think about the whole grid. Think about the **last cell** — the bottom-right. You arrived there from exactly one of two neighbors: the cell **above** it, or the cell **to its left**. That's the entire decision structure.

Translate that into: _the number of paths to reach cell $(m, n)$ equals paths-to-the-cell-above plus paths-to-the-cell-to-the-left._ Now figure out what "the cell above" and "the cell to the left" mean in terms of $m$ and $n$, and what the smallest solvable grid is.

Take your time.


In [1]:
def count_paths(m: int, n: int) -> int:
    # number of distinct paths from top-left to bottom-right
    if m == 0 or n == 0:
        return 0
    if m == 1 or n == 1:
        return 1
    return count_paths(m - 1, n) + count_paths(m, n - 1)

In [2]:
count_paths(2, 2)

2

# count_paths — key points from Claude's Feedback

## Part 1: The recurrence and base cases

**Problem.** In a grid with $m$ rows and $n$ columns, start at the top-left cell, move only right or down, and count the distinct paths to the bottom-right cell. Here $m$ and $n$ are the grid's dimensions, not cell coordinates.

**Recurrence.** The last move into the bottom-right cell came either from the cell above it (a grid with one fewer row) or from the cell to its left (a grid with one fewer column). So:
$f(m, n) = f(m-1, n) + f(m, n-1)$

**Base cases.**

- $f(m, 0) = f(0, n) = 0$. A grid with a zero dimension has no cells, so there is no path.
- $f(1, n) = f(m, 1) = 1$. With a single row you can only move right, and with a single column you can only move down, so there is exactly one path.

**My solution was correct by construction**, not correct by accident.

**The load-bearing test.** To check whether a special case is a real base case or a patch, delete it and trace the smallest input.

- If I delete `m == 1 or n == 1`, then $f(1,1) = f(0,1) + f(1,0) = 0 + 0 = 0$, which is wrong. The recurrence cannot rebuild this case from the zero case, so it is a **real base case**.
- In climb_stairs, the `n == 1` case was a **patch**. It was hiding a wrong base case (negative steps returning 1 instead of 0). Once both base cases were individually true, `n == 1` could be deleted and the recurrence still gave the right answer.

**Style notes.**

- Prefer `if m == 0 or n == 0` over `if not m or not n`. `not m` is also true for `None` and `0.0`, and it hides the real condition. Write code that states the invariant literally, which here is "a dimension is zero".
- Order base cases from most degenerate to least degenerate: zero dimension first, then single row or column. This prevents bugs when the cases overlap.

**Math connection.** $f(m, n) = \binom{(m-1)+(n-1)}{m-1}$. The grid recurrence is Pascal's triangle turned on its side. For example, $f(2,3) = \binom{3}{1} = 3$.

## Part 2: Why $f(1,1) = 1$

- A path is a **sequence of moves** from the start cell to the destination cell.
- In a 1×1 grid the start cell is the destination cell, so the only path is the **empty sequence** (zero moves). There is exactly one empty sequence, so the count is 1.
- Keep the two questions separate. "How many moves?" has the answer 0. "How many distinct paths?" has the answer 1. The function answers the second question.
- The recurrence forces this value. $f(1,2) = f(0,2) + f(1,1) = 0 + f(1,1)$, and a 1×2 grid clearly has 1 path, so $f(1,1)$ must be 1.
- RL link: the value of the "already at the goal" state is set by whatever keeps the recurrence consistent. For counting problems that value is 1 (the empty path). In the Bellman equation it is the terminal reward. If this base case is wrong, every value built on top of it is silently wrong.

## Part 3: Overlapping subproblems and how counting works

**Two properties make a problem a DP candidate.**

- **Optimal substructure:** the big answer is built from answers to smaller subproblems. The recurrence above is this property.
- **Overlapping subproblems:** the same subproblem is reached by more than one route. For example, $(3,3)$ is reached from $(3,4)$ (first move was Down) and from $(4,3)$ (first move was Right).

**Used twice is correct counting. Computed twice is wasted work.**

- The paths through $(3,4)$ all start with a Down move. The paths through $(4,3)$ all start with a Right move. These two sets of paths are **disjoint**, so adding them never double-counts a path.
- The value $f(3,3) = 6$ is **used** in both branches. That is correct, because it contributes to two different families of full paths.
- Plain recursion **computes** $f(3,3)$ from scratch every time it reaches it. That is wasteful, because the answer for the same arguments never depends on how you arrived there.
- DP keeps the correct double use and removes the wasteful double computation.

**Dry run of $f(4,4)$.** Fill each cell as the cell above plus the cell to the left. The first row and first column are base cases (all 1).

|         | n=1 | n=2 | n=3 | n=4    |
| ------- | --- | --- | --- | ------ |
| **m=1** | 1   | 1   | 1   | 1      |
| **m=2** | 1   | 2   | 3   | 4      |
| **m=3** | 1   | 3   | 6   | 10     |
| **m=4** | 1   | 4   | 10  | **20** |

- $f(4,4) = f(3,4) + f(4,3) = 10 + 10 = 20$. The first 10 is all paths whose first move is Down. The second 10 is all paths whose first move is Right.
- Filling this table from the base cases outward is **bottom-up DP**. A memoized recursion (**top-down DP**) computes the same numbers, but only when they are first needed.

**Cost.** There are at most $m \times n$ distinct subproblems.

- Without a memo the work is exponential, because the tree branches twice at almost every node.
- With a memo each subproblem is computed once, so the work is about $m \times n$.

## Next: add the memo (Option A)

- The outer `count_paths(m, n)` creates `memo = {}`, and an inner `helper(m, n)` does the recursion.
- Key the memo on the tuple `(m, n)`. A tuple is hashable, and a list is not.
- Invariant: check the memo before computing. On a miss, compute with the recurrence, store the result, then return it. Every stored value is final and never changes.
- A fresh memo for each top-level call means entries from one grid can never leak into a call for another grid.
